# 03  --  Time Series Analysis

This notebook examines how EU energy production has changed over time using SQL window functions.
Rather than snapshots, we track trends  --  year-over-year growth, smoothed trajectories, and cumulative shifts.

**Central question:** Is the EU's green energy transition real, or is renewable share growth masking continued fossil fuel dependence?

---

**SQL window functions used:**

| Section | Function | Purpose |
|---|---|---|
| 1 | `LAG()` | Compare each year to the prior year |
| 2 | `AVG() OVER (ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` | 3-year rolling average |
| 3 -- 4 | `FIRST_VALUE()` | Index values to a 2005 baseline |
| 5 | `RANK() OVER (PARTITION BY year)` | EU ranking each year by renewable share |

In [1]:
import pandas as pd
import sqlite3
from IPython.display import display
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from utils.config import DB_PATH
from utils.charts import (
    chart_yoy_renewable_growth,
    chart_yoy_fossil_vs_renewable,
    chart_rolling_dependency,
    chart_rolling_renewable_share,
    chart_germany_indexed,
    chart_germany_gap,
    chart_archetype_renewable_share,
    chart_archetype_dependency,
    chart_archetype_rank,
    chart_eu_renewable_share_trend,
    chart_absolute_fossil_consumption,
    chart_country_scorecard,
)

conn = sqlite3.connect(DB_PATH)

## 1. Year-over-Year Changes

`LAG(value) OVER (ORDER BY year)` looks back one row in a sorted result set, returning the previous year's value.
Dividing the year-on-year difference by the previous year's value gives the percentage growth rate.

Because we want EU-level totals rather than country-level changes, we aggregate by year in a CTE first,
then apply `LAG()` on the aggregated result  --  window functions run after `GROUP BY`.

**Questions:** Is renewable production consistently growing? Are fossil fuels declining at a matching rate?

In [2]:
# EU-wide YoY % change in renewable primary production
df = pd.read_sql("""
    WITH eu_totals AS (
        SELECT
            year,
            SUM(value_gwh) AS total_renewable
        FROM renewables
        WHERE energy_source = 'Renewables and biofuels'
          AND balance_type = 'Primary production'
        GROUP BY year
    )
    SELECT
        year,
        ROUND(
            (total_renewable - LAG(total_renewable) OVER (ORDER BY year))
            / LAG(total_renewable) OVER (ORDER BY year) * 100,
            2
        ) AS yoy_change_pct
    FROM eu_totals
    ORDER BY year
""", conn)
display(df)

,year,yoy_change_pct
0,2005,NaN
1,2006,5.92
2,2007,7.74
3,2008,7.69
4,2009,4.27
5,2010,11.35
6,2011,-1.93
7,2012,10.97
8,2013,6.08
9,2014,-0.05


In [3]:
fig = chart_yoy_renewable_growth(df)
fig.show()

In [4]:
# EU-wide YoY % change: fossil fuel gross available energy vs renewable production
# Two separate CTEs joined on year, then LAG() applied to each series
df = pd.read_sql("""
    WITH fossil_totals AS (
        SELECT year, SUM(value_gwh) AS total_fossil
        FROM fossil_fuels
        WHERE balance_type = 'Gross available energy'
        GROUP BY year
    ),
    renewable_totals AS (
        SELECT year, SUM(value_gwh) AS total_renewable
        FROM renewables
        WHERE energy_source = 'Renewables and biofuels'
          AND balance_type = 'Primary production'
        GROUP BY year
    )
    SELECT
        f.year,
        ROUND(
            (f.total_fossil - LAG(f.total_fossil) OVER (ORDER BY f.year))
            / LAG(f.total_fossil) OVER (ORDER BY f.year) * 100,
            2
        ) AS fossil_yoy_pct,
        ROUND(
            (r.total_renewable - LAG(r.total_renewable) OVER (ORDER BY r.year))
            / LAG(r.total_renewable) OVER (ORDER BY r.year) * 100,
            2
        ) AS renewable_yoy_pct
    FROM fossil_totals f
    JOIN renewable_totals r ON f.year = r.year
    ORDER BY f.year
""", conn)
display(df)

,year,fossil_yoy_pct,renewable_yoy_pct
0,2005,NaN,NaN
1,2006,0.69,5.92
2,2007,-1.60,7.74
3,2008,-1.44,7.69
4,2009,-7.26,4.27
5,2010,2.98,11.35
6,2011,-3.28,-1.93
7,2012,-2.88,10.97
8,2013,-2.18,6.08
9,2014,-5.02,-0.05


In [5]:
fig = chart_yoy_fossil_vs_renewable(df)
fig.show()

## 2. Rolling Averages

Year-on-year data is noisy  --  COVID caused a sharp demand drop in 2020, the 2022 gas crisis briefly shifted the mix.
A **3-year rolling average** smooths these shocks to reveal the underlying trend.

`AVG(value) OVER (ORDER BY year ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)` includes the current row and the two rows before it.
Adding `PARTITION BY country` gives each country its own independent rolling window.

We apply this to two metrics: the EU average dependency rate and renewable share across four archetype countries.

In [6]:
# EU average energy dependency rate per year + 3-year rolling average
# Step 1: compute dependency rate per country (CASE WHEN pivot)
# Step 2: average across countries per year
# Step 3: apply rolling window on the averaged result
df = pd.read_sql("""
    WITH country_dep AS (
        SELECT
            year,
            country,
            ROUND(
                (SUM(CASE WHEN balance_type = 'Imports' THEN value_gwh ELSE 0 END)
                 - SUM(CASE WHEN balance_type = 'Exports' THEN value_gwh ELSE 0 END))
                / NULLIF(SUM(CASE WHEN balance_type = 'Gross available energy' THEN value_gwh ELSE 0 END), 0) * 100,
                2
            ) AS dependency_rate
        FROM energy_dependency
        WHERE energy_source = 'Total'
        GROUP BY year, country
    ),
    eu_avg AS (
        SELECT
            year,
            ROUND(AVG(dependency_rate), 2) AS avg_dependency
        FROM country_dep
        GROUP BY year
    )
    SELECT
        year,
        avg_dependency,
        ROUND(
            AVG(avg_dependency) OVER (ORDER BY year ROWS BETWEEN 2 PRECEDING AND CURRENT ROW),
            2
        ) AS rolling_avg
    FROM eu_avg
    ORDER BY year
""", conn)
display(df)

,year,avg_dependency,rolling_avg
0,2005,57.45,57.45
1,2006,58.52,57.98
2,2007,57.95,57.97
3,2008,58.87,58.45
4,2009,56.72,57.85
5,2010,55.89,57.16
6,2011,56.90,56.50
7,2012,56.02,56.27
8,2013,55.50,56.14
9,2014,55.01,55.51


In [7]:
fig = chart_rolling_dependency(df)
fig.show()

In [8]:
# 3-year rolling average of renewable share for Germany, France, Italy, Denmark
# PARTITION BY country ensures each country gets its own independent rolling window
df = pd.read_sql("""
    WITH country_renew AS (
        SELECT year, country, SUM(value_gwh) AS renewable_gwh
        FROM renewables
        WHERE energy_source = 'Renewables and biofuels'
          AND balance_type = 'Primary production'
        GROUP BY year, country
    ),
    country_gae AS (
        SELECT year, country, SUM(value_gwh) AS gae
        FROM energy_dependency
        WHERE balance_type = 'Gross available energy'
          AND energy_source = 'Total'
        GROUP BY year, country
    ),
    shares AS (
        SELECT
            r.year,
            r.country,
            ROUND(r.renewable_gwh / NULLIF(g.gae, 0) * 100, 2) AS renewable_share
        FROM country_renew r
        JOIN country_gae g ON r.year = g.year AND r.country = g.country
    )
    SELECT
        year,
        country,
        ROUND(
            AVG(renewable_share) OVER (PARTITION BY country ORDER BY year ROWS BETWEEN 2 PRECEDING AND CURRENT ROW),
            2
        ) AS rolling_share
    FROM shares
    WHERE country IN ('Germany', 'France', 'Italy', 'Denmark')
    ORDER BY country, year
""", conn)
display(df)

,year,country,rolling_share
0,2005,Denmark,12.31
1,2006,Denmark,11.93
2,2007,Denmark,12.33
3,2008,Denmark,12.62
4,2009,Denmark,13.44
...,...,...,...
75,2020,Italy,17.56
76,2021,Italy,17.95
77,2022,Italy,17.92
78,2023,Italy,17.71


In [9]:
fig = chart_rolling_renewable_share(df, ['Germany', 'France', 'Italy', 'Denmark'])
fig.show()

## 3. Germany: The Nuclear Phaseout Story

Germany shut down its last three nuclear plants in April 2023, completing a phaseout that began after Fukushima in 2011.
At peak (2000), nuclear supplied ~30% of Germany's electricity. By 2024, it was zero.

To compare nuclear decline and renewable growth on the same scale, we index all metrics to **2005 = 100** using `FIRST_VALUE()`.

`FIRST_VALUE(value) OVER (ORDER BY year)` returns the first value in the ordered window  --  dividing every
subsequent row by it converts absolute GWh into a relative index, making otherwise incomparable series directly comparable.

In [10]:
# Germany: nuclear, renewables, fossil imports, dependency rate â€” all indexed to 2005 = 100
# Four CTEs pull one metric each; combined CTE joins them; outer SELECT applies FIRST_VALUE()
df = pd.read_sql("""
    WITH nuclear AS (
        SELECT year, value_gwh AS nuclear_gwh
        FROM renewables
        WHERE country = 'Germany'
          AND energy_source = 'Nuclear heat'
          AND balance_type = 'Primary production'
    ),
    renew AS (
        SELECT year, value_gwh AS renewable_gwh
        FROM renewables
        WHERE country = 'Germany'
          AND energy_source = 'Renewables and biofuels'
          AND balance_type = 'Primary production'
    ),
    fossil AS (
        SELECT year, SUM(value_gwh) AS fossil_gwh
        FROM fossil_fuels
        WHERE country = 'Germany'
          AND balance_type = 'Gross available energy'
        GROUP BY year
    ),
    dep AS (
        SELECT
            year,
            ROUND(
                (SUM(CASE WHEN balance_type = 'Imports' THEN value_gwh ELSE 0 END)
                 - SUM(CASE WHEN balance_type = 'Exports' THEN value_gwh ELSE 0 END))
                / NULLIF(SUM(CASE WHEN balance_type = 'Gross available energy' THEN value_gwh ELSE 0 END), 0) * 100,
                2
            ) AS dependency_rate
        FROM energy_dependency
        WHERE country = 'Germany'
          AND energy_source = 'Total'
        GROUP BY year
    ),
    combined AS (
        SELECT n.year, n.nuclear_gwh, r.renewable_gwh, f.fossil_gwh, d.dependency_rate
        FROM nuclear n
        JOIN renew r  ON n.year = r.year
        JOIN fossil f ON n.year = f.year
        JOIN dep d    ON n.year = d.year
    )
    SELECT
        year,
        ROUND(nuclear_gwh     / FIRST_VALUE(nuclear_gwh)     OVER (ORDER BY year) * 100, 1) AS nuclear_idx,
        ROUND(renewable_gwh   / FIRST_VALUE(renewable_gwh)   OVER (ORDER BY year) * 100, 1) AS renewable_idx,
        ROUND(fossil_gwh      / FIRST_VALUE(fossil_gwh)      OVER (ORDER BY year) * 100, 1) AS fossil_idx,
        ROUND(dependency_rate / FIRST_VALUE(dependency_rate) OVER (ORDER BY year) * 100, 1) AS dependency_idx
    FROM combined
    ORDER BY year
""", conn)
display(df)

,year,nuclear_idx,renewable_idx,fossil_idx,dependency_idx
0,2005,100.0,100.0,100.0,100.0
1,2006,102.6,122.0,102.2,100.4
2,2007,86.2,142.2,97.2,96.3
3,2008,91.1,149.4,97.5,100.0
4,2009,82.6,149.7,90.5,100.6
5,2010,86.1,171.9,94.5,98.8
6,2011,66.1,178.5,90.6,101.7
7,2012,60.9,199.8,91.3,100.8
8,2013,59.6,206.9,93.8,102.7
9,2014,59.5,207.7,88.9,101.7


In [11]:
fig = chart_germany_indexed(df)
fig.show()

## 4. Did Renewables Cover the Nuclear Gap?

Germany lost roughly **0.49 million GWh** of nuclear production between 2005 and 2024.
The question is whether the renewable build-out covered that loss, or whether the gap was filled by fossil fuel imports.

We compute **cumulative change** from the 2005 baseline by:
1. Using `LAG()` to get the year-on-year change for each metric
2. Using `SUM() OVER (ORDER BY year)` to accumulate those changes from the first year onwards

The result shows, at each point in time, how much nuclear production Germany had lost and how much renewable production it had gained  --  in the same unit (million GWh).

> **Note:** `nuclear_loss_cumulative` is negative in the raw data  --  nuclear declined, so the cumulative change is negative. The chart displays its absolute value.

In [12]:
# Germany: cumulative nuclear loss vs cumulative renewable gain from 2005 baseline
df = pd.read_sql("""
    WITH germany_annual AS (
        SELECT
            year,
            SUM(CASE WHEN energy_source = 'Nuclear heat'            THEN value_gwh ELSE 0 END) AS nuclear_gwh,
            SUM(CASE WHEN energy_source = 'Renewables and biofuels' THEN value_gwh ELSE 0 END) AS renewable_gwh
        FROM renewables
        WHERE country = 'Germany'
          AND balance_type = 'Primary production'
        GROUP BY year
    ),
    annual_changes AS (
        SELECT
            year,
            nuclear_gwh   - LAG(nuclear_gwh)   OVER (ORDER BY year) AS nuclear_yoy,
            renewable_gwh - LAG(renewable_gwh) OVER (ORDER BY year) AS renewable_yoy
        FROM germany_annual
    )
    SELECT
        year,
        ROUND(SUM(nuclear_yoy)   OVER (ORDER BY year) / 1e6, 3) AS nuclear_loss_cumulative,
        ROUND(SUM(renewable_yoy) OVER (ORDER BY year) / 1e6, 3) AS renewable_gain_cumulative
    FROM annual_changes
    ORDER BY year
""", conn)
display(df)

,year,nuclear_loss_cumulative,renewable_gain_cumulative
0,2005,NaN,NaN
1,2006,0.013,0.046
2,2007,-0.068,0.088
3,2008,-0.044,0.103
4,2009,-0.085,0.104
5,2010,-0.068,0.150
6,2011,-0.166,0.164
7,2012,-0.191,0.208
8,2013,-0.198,0.223
9,2014,-0.198,0.225


In [13]:
fig = chart_germany_gap(df)
fig.show()

## 5. Four Country Archetypes

Energy policy differs sharply across the EU. Four countries illustrate distinct paths:

| Country | Nuclear | Renewable strategy |
|---|---|---|
| **Germany** | Full phaseout by 2023 | Heavy wind and solar investment |
| **France** | Retained (~70% electricity share) | Slow renewable build-out |
| **Italy** | None since 1990 referendum | Gradual solar growth |
| **Denmark** | Never had nuclear | Wind pioneer  --  among EU's highest renewable shares |

`RANK() OVER (PARTITION BY year ORDER BY renewable_share DESC)` assigns each country an EU-wide rank every year.
`PARTITION BY year` resets the ranking for each year independently, so we can track whether each country moved up or down relative to its peers over time.

In [14]:
# Renewable share per year for the four archetype countries
df = pd.read_sql("""
    WITH country_renew AS (
        SELECT year, country, SUM(value_gwh) AS renewable_gwh
        FROM renewables
        WHERE energy_source = 'Renewables and biofuels'
          AND balance_type = 'Primary production'
        GROUP BY year, country
    ),
    country_gae AS (
        SELECT year, country, SUM(value_gwh) AS gae
        FROM energy_dependency
        WHERE balance_type = 'Gross available energy'
          AND energy_source = 'Total'
        GROUP BY year, country
    )
    SELECT
        r.year,
        r.country,
        ROUND(r.renewable_gwh / NULLIF(g.gae, 0) * 100, 2) AS renewable_share
    FROM country_renew r
    JOIN country_gae g ON r.year = g.year AND r.country = g.country
    WHERE r.country IN ('Germany', 'France', 'Italy', 'Denmark')
    ORDER BY r.country, r.year
""", conn)
display(df)

,year,country,renewable_share
0,2005,Denmark,12.31
1,2006,Denmark,11.55
2,2007,Denmark,13.14
3,2008,Denmark,13.17
4,2009,Denmark,14.00
...,...,...,...
75,2020,Italy,18.85
76,2021,Italy,17.86
77,2022,Italy,17.05
78,2023,Italy,18.23


In [15]:
fig = chart_archetype_renewable_share(df)
fig.show()

In [16]:
# Energy dependency rate per year for the four countries
df = pd.read_sql("""
    SELECT
        year,
        country,
        ROUND(
            (SUM(CASE WHEN balance_type = 'Imports' THEN value_gwh ELSE 0 END)
             - SUM(CASE WHEN balance_type = 'Exports' THEN value_gwh ELSE 0 END))
            / NULLIF(SUM(CASE WHEN balance_type = 'Gross available energy' THEN value_gwh ELSE 0 END), 0) * 100,
            2
        ) AS dependency_rate
    FROM energy_dependency
    WHERE energy_source = 'Total'
      AND country IN ('Germany', 'France', 'Italy', 'Denmark')
    GROUP BY year, country
    ORDER BY country, year
""", conn)
display(df)

,year,country,dependency_rate
0,2005,Denmark,-50.62
1,2006,Denmark,-35.49
2,2007,Denmark,-24.25
3,2008,Denmark,-21.13
4,2009,Denmark,-18.80
...,...,...,...
75,2020,Italy,73.45
76,2021,Italy,73.42
77,2022,Italy,79.49
78,2023,Italy,75.36


In [17]:
fig = chart_archetype_dependency(df)
fig.show()

In [18]:
# EU rank by renewable share for the four countries, each year
# RANK() OVER (PARTITION BY year) resets the ranking independently per year
df = pd.read_sql("""
    WITH country_renew AS (
        SELECT year, country, SUM(value_gwh) AS renewable_gwh
        FROM renewables
        WHERE energy_source = 'Renewables and biofuels'
          AND balance_type = 'Primary production'
        GROUP BY year, country
    ),
    country_gae AS (
        SELECT year, country, SUM(value_gwh) AS gae
        FROM energy_dependency
        WHERE balance_type = 'Gross available energy'
          AND energy_source = 'Total'
        GROUP BY year, country
    ),
    shares AS (
        SELECT
            r.year,
            r.country,
            ROUND(r.renewable_gwh / NULLIF(g.gae, 0) * 100, 2) AS renewable_share
        FROM country_renew r
        JOIN country_gae g ON r.year = g.year AND r.country = g.country
    ),
    ranked AS (
        SELECT
            year,
            country,
            RANK() OVER (PARTITION BY year ORDER BY renewable_share DESC) AS rank
        FROM shares
    )
    SELECT year, country, rank
    FROM ranked
    WHERE country IN ('Germany', 'France', 'Italy', 'Denmark')
    ORDER BY country, year
""", conn)
display(df)

,year,country,rank
0,2005,Denmark,10
1,2006,Denmark,9
2,2007,Denmark,7
3,2008,Denmark,9
4,2009,Denmark,10
...,...,...,...
75,2020,Italy,10
76,2021,Italy,10
77,2022,Italy,12
78,2023,Italy,13


In [19]:
fig = chart_archetype_rank(df)
fig.show()

## 6. EU Green Energy Reality Check

Politicians and press releases cite renewable **share**  --  the fraction of total energy that comes from renewables.
But share can grow simply because total energy consumption falls, without any real increase in clean energy.

We look at two metrics in parallel:
- **Renewable share (%)**  --  the headline figure
- **Absolute fossil fuel consumption (million GWh)**  --  what the climate actually responds to

If fossil consumption is flat or rising while renewable share grows, renewables are supplementing energy demand  --  not replacing fossil fuels.

The country scorecard at the end ranks all 27 EU members by how much their renewable share grew from 2005 to 2024,
using conditional aggregation (`MAX(CASE WHEN year = ...)`) to pivot the first and last year into columns without a self-join.

In [20]:
# EU renewable share of gross available energy + 3-year rolling average
df = pd.read_sql("""
    WITH eu_renew AS (
        SELECT year, SUM(value_gwh) AS renewable_gwh
        FROM renewables
        WHERE energy_source = 'Renewables and biofuels'
          AND balance_type = 'Primary production'
        GROUP BY year
    ),
    eu_gae AS (
        SELECT year, SUM(value_gwh) AS gae
        FROM energy_dependency
        WHERE balance_type = 'Gross available energy'
          AND energy_source = 'Total'
        GROUP BY year
    ),
    shares AS (
        SELECT
            r.year,
            ROUND(r.renewable_gwh / NULLIF(g.gae, 0) * 100, 2) AS renewable_share
        FROM eu_renew r
        JOIN eu_gae g ON r.year = g.year
    )
    SELECT
        year,
        renewable_share,
        ROUND(
            AVG(renewable_share) OVER (ORDER BY year ROWS BETWEEN 2 PRECEDING AND CURRENT ROW),
            2
        ) AS rolling_share
    FROM shares
    ORDER BY year
""", conn)
display(df)

,year,renewable_share,rolling_share
0,2005,7.16,7.16
1,2006,7.51,7.33
2,2007,8.18,7.62
3,2008,8.84,8.18
4,2009,9.81,8.94
5,2010,10.51,9.72
6,2011,10.61,10.31
7,2012,11.94,11.02
8,2013,12.81,11.79
9,2014,13.26,12.67


In [21]:
fig = chart_eu_renewable_share_trend(df)
fig.show()

In [22]:
# EU absolute fossil fuel gross available energy (million GWh)
df = pd.read_sql("""
    SELECT
        year,
        ROUND(SUM(value_gwh) / 1e6, 3) AS fossil_gwh_m
    FROM fossil_fuels
    WHERE balance_type = 'Gross available energy'
    GROUP BY year
    ORDER BY year
""", conn)
display(df)

,year,fossil_gwh_m
0,2005,14.884
1,2006,14.986
2,2007,14.747
3,2008,14.534
4,2009,13.478
5,2010,13.880
6,2011,13.425
7,2012,13.038
8,2013,12.754
9,2014,12.113


In [23]:
fig = chart_absolute_fossil_consumption(df)
fig.show()

In [24]:
# Per-country renewable share growth: 2005 baseline vs latest year
# Conditional aggregation (MAX + CASE WHEN) pivots rows into columns without a self-join
df = pd.read_sql("""
    WITH country_renew AS (
        SELECT year, country, SUM(value_gwh) AS renewable_gwh
        FROM renewables
        WHERE energy_source = 'Renewables and biofuels'
          AND balance_type = 'Primary production'
        GROUP BY year, country
    ),
    country_gae AS (
        SELECT year, country, SUM(value_gwh) AS gae
        FROM energy_dependency
        WHERE balance_type = 'Gross available energy'
          AND energy_source = 'Total'
        GROUP BY year, country
    ),
    shares AS (
        SELECT
            r.year,
            r.country,
            ROUND(r.renewable_gwh / NULLIF(g.gae, 0) * 100, 2) AS renewable_share
        FROM country_renew r
        JOIN country_gae g ON r.year = g.year AND r.country = g.country
    )
    SELECT
        country,
        MAX(CASE WHEN year = (SELECT MIN(year) FROM shares) THEN renewable_share END) AS share_2005,
        MAX(CASE WHEN year = (SELECT MAX(year) FROM shares) THEN renewable_share END) AS share_2024,
        ROUND(
            MAX(CASE WHEN year = (SELECT MAX(year) FROM shares) THEN renewable_share END)
            - MAX(CASE WHEN year = (SELECT MIN(year) FROM shares) THEN renewable_share END),
            2
        ) AS growth
    FROM shares
    GROUP BY country
    HAVING share_2005 IS NOT NULL AND share_2024 IS NOT NULL
    ORDER BY growth DESC
""", conn)
conn.close()
display(df)

,country,share_2005,share_2024,growth
0,Estonia,12.33,45.45,33.12
1,Latvia,38.23,69.88,31.65
2,Portugal,12.40,34.52,22.12
3,Lithuania,9.86,29.88,20.02
4,Sweden,27.09,46.96,19.87
5,Denmark,12.31,30.61,18.30
6,Austria,20.96,39.05,18.09
7,Finland,23.09,40.27,17.18
8,Germany,5.15,20.02,14.87
9,Spain,5.51,18.56,13.05


In [25]:
fig = chart_country_scorecard(df)
fig.show()

## Summary

**Year-over-year changes (Section 1)**
EU renewable production grew in 16 out of 19 years. The three contractions were 2011 (-1.93%), 2014 (-0.05%), and 2022 (-0.61%), with 2022 driven by drought cutting hydro output during the same year as the gas crisis.
Fossil fuel consumption shows the opposite pattern: declining in most years since 2008, with the sharpest drops around COVID (2020) and the post-Ukraine energy squeeze (2022-2023).
The two trends are not symmetric -- renewable growth averages ~4% per year while fossil decline averages ~2%, meaning the transition is real but gradual.

**Rolling averages (Section 2)**
Smoothing removes single-year noise. The EU average dependency rate rose from 2005 to 2008, fell through 2014, crept back up to a spike in 2019, dipped in 2020-2021, spiked again in 2022, and has been declining since.
Across the four archetype countries, the rolling renewable share trajectories diverge sharply: Denmark leads with the highest share among the four and a steady upward trend; Germany climbed past Italy and is continuing to rise; Italy is stable but not catching up; France holds steady at the lowest share of the four due to its nuclear-heavy mix. Notably, Denmark's dependency rate tells a different story -- it swung from -51% in 2005 (a large net exporter, driven by North Sea oil and gas) to +38% in 2024 (a net importer), a reversal explored further in Section 5.

**Germany: phaseout in context (Section 3)**
With everything indexed to 2005 = 100, nuclear reached zero by 2024, renewables grew to roughly 2.8x their 2005 level (~280 index), and fossil imports declined to approximately 70% of the 2005 baseline. Despite this investment, Germany's dependency rate ended at 110 -- 10% above the 2005 baseline -- meaning the phaseout permanently raised Germany's import exposure rather than the renewable build-out compensating for it.

**The compensation question (Section 4)**
Comparing cumulative renewable gain against cumulative nuclear loss, renewable gains exceeded nuclear losses in most years. The exceptions were 2011, 2022, 2023, and 2024 -- years when accelerated shutdowns or crisis-driven demand shifts caused the nuclear loss to briefly outpace what renewables had gained. By 2024 the gap had widened again: renewables had gained 0.38 million GWh cumulatively while nuclear losses totalled 0.49 million GWh, leaving a ~0.11 million GWh gap covered by fossil imports.

**Four country archetypes (Section 5)**
Looking at EU rank by renewable share: Denmark averaged around 8th place throughout the period (range: 6th-11th), a strong performer but not the absolute EU leader. Germany improved from 17th in 2005 to 10th by 2024, overtaking Italy in the process. Italy held a stable rank around 11th-12th. France declined from 14th in 2005 to consistently around 18th-21st by the end of the period -- its static nuclear strategy left it falling behind as other countries built out renewables. Denmark's rank is worth pairing with its dependency chart: it improved its renewable rank while its import dependency swung 89 percentage points in the wrong direction, a reminder that renewable share and energy security are related but distinct metrics.

**EU green energy reality check (Section 6)**
EU renewable share grew from 7.2% in 2005 to 19.8% by 2024, and absolute fossil consumption fell from ~15 to ~10.4 million GWh -- confirming real displacement rather than statistical dilution. Every EU country recorded renewable share growth; Estonia led with +33pp, Latvia +32pp, Germany 9th at +15pp.
The dependency picture is more sobering. Despite the renewable buildout, EU average import dependency fell only from 57.5% in 2005 to 56.0% in 2024 -- 1.5 percentage points over nearly two decades. Dependency is driven by many factors simultaneously: domestic fossil production levels, nuclear policy decisions, total demand trends, and trade patterns. Countries that expanded renewables while closing nuclear plants often found the two effects largely cancelled out. The data confirms the transition is real; whether it is sufficient, or whether a different policy mix would have reduced dependency more significantly, remains an open question.

In [26]:
conn.close()